# Part 2 — Ensemble M-sweep, MC-Dropout, member-level TS
**Calibration vs Deep Ensembles, MAKE revision**

## What gets computed

| Model | Calibrators | Rows |
|---|---|---|
| `deep_ensemble_m3` (new) | none, temp, logistic, isotonic, dirichlet, temp_member | 6 × 5 × 36 = 1,080 |
| `deep_ensemble_m10` (new) | same 6 | 1,080 |
| `deep_ensemble` (M=5, existing) | **only** temp_member (regular 5 already in Part 1) | 1 × 5 × 36 = 180 |
| `mc_dropout` (new) | none, temp, logistic, isotonic, dirichlet | 5 × 5 × 36 = 900 |
| **Total new rows** | | **3,240** |

Final CSV: **7,740 rows** 

## 1. Environment setup

In [ ]:
# Working directory for cached probs and intermediate CSVs.
# Override with:  export WORK_DIR=/path/to/persistent/storage
import os, pathlib

WORK_DIR = pathlib.Path(os.environ.get('WORK_DIR', './work')).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
PROBS_DIR = WORK_DIR / 'probs'
PROBS_DIR.mkdir(exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print(f"Cached probability files: {len(list(PROBS_DIR.glob('*.npz')))}")


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    print("  TF32 + cudnn.benchmark enabled.")
else:
    print("WARNING: no GPU detected.")

In [ ]:
# Install dependencies
!pip install catboost dirichletcal
!pip install openml==0.15.1 lightgbm==4.3.0 'xgboost>=2.0.0' 'scikit-learn>=1.3.0'
import catboost, dirichletcal
print(f"✓ catboost {catboost.__version__}, dirichletcal {dirichletcal.__version__}")

## 2. Get your codebase + existing artifacts

In [ ]:
# Locate the repo root (the directory containing src/) and add it to sys.path.
# Works regardless of whether the notebook is run from notebooks/, the repo root,
# or one level deeper.
import os, pathlib, sys

def _find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'src').is_dir():
            return cand
    raise FileNotFoundError(
        "Could not find a 'src/' directory in the current working directory or any "
        "parent. Run this notebook from inside the cloned repository."
    )

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")
print("✓ src/ found")


In [ ]:
import sys
sys.path.insert(0, os.getcwd())
from src.datasets import load_task, split_dataset
from src.calibration import build_calibrator, CALIBRATOR_LABELS, BaseCalibrator, TemperatureScaling
from src.metrics import evaluate_all
from src.models import LightGBMModel, XGBoostModel, SingleMLP, DeepEnsemble, get_device
from src.utils import EPS
print("✓ src/ imports OK")

In [ ]:
# Verify prerequisites: dataset_manifest.csv and (optionally) results_raw.csv from notebook 01.
import pathlib, shutil
import pandas as pd

# Manifest: required.
manifest_candidates = [
    WORK_DIR / 'dataset_manifest.csv',
    REPO_ROOT / 'data' / 'dataset_manifest.csv',
]
manifest_path = next((p for p in manifest_candidates if p.exists()), None)
if manifest_path is None:
    raise FileNotFoundError(
        "dataset_manifest.csv not found. Place it in $WORK_DIR or in the repo's "
        f"data/ directory. Searched: {[str(p) for p in manifest_candidates]}"
    )
if manifest_path != WORK_DIR / 'dataset_manifest.csv':
    shutil.copy(manifest_path, WORK_DIR / 'dataset_manifest.csv')

manifest = pd.read_csv(WORK_DIR / 'dataset_manifest.csv')
print(f"✓ Manifest: {len(manifest)} datasets")

# Existing results from notebook 01: required by this notebook.
results_path = WORK_DIR / 'results_raw.csv'
if not results_path.exists():
    raise FileNotFoundError(
        f"{results_path} not found. Run notebook 01 first, or extract the cached "
        "results archive into $WORK_DIR."
    )

existing = pd.read_csv(results_path)
print(f"✓ Existing results: {len(existing)} rows")
print(f"  models: {sorted(existing['model'].unique())}")
print(f"  calibrators: {sorted(existing['calibrator'].unique())}")


## 3. Re-define CatBoost and Dirichlet

Needed only so the train-once-calibrate-many runner is self-contained and can re-derive rows if probs are missing for some reason.

In [ ]:
import numpy as np
from catboost import CatBoostClassifier

class CatBoostModel:
    _DEFAULT_PARAMS = {
        "learning_rate": 0.05, "depth": 6, "min_data_in_leaf": 20,
        "rsm": 0.8, "subsample": 0.8, "bootstrap_type": "Bernoulli",
        "l2_leaf_reg": 3.0, "od_type": "Iter", "od_wait": 50,
        "verbose": False, "thread_count": -1, "allow_writing_files": False,
    }
    def __init__(self, n_classes, num_boost_round=500, extra_params=None):
        self.n_classes = n_classes
        params = dict(self._DEFAULT_PARAMS)
        params["iterations"] = num_boost_round
        if n_classes > 2:
            params.update({"loss_function":"MultiClass","eval_metric":"MultiClass","classes_count":n_classes})
        else:
            params.update({"loss_function":"Logloss","eval_metric":"Logloss"})
        if extra_params: params.update(extra_params)
        self._params = params
        self._model = None
    def fit(self, X_train, y_train, X_val=None, y_val=None):
        self._model = CatBoostClassifier(**self._params)
        eval_set = (X_val, y_val) if X_val is not None else None
        self._model.fit(X_train, y_train, eval_set=eval_set, verbose=False, plot=False)
        return self
    def predict_proba(self, X):
        return np.clip(self._model.predict_proba(X), EPS, 1.0)

from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

class DirichletODIRCalibrator(BaseCalibrator):
    def __init__(self, reg_lambda=1e-3, reg_mu=1e-3):
        self.reg_lambda = reg_lambda; self.reg_mu = reg_mu; self._model = None
    def fit(self, probs_val, y_val):
        self._model = FullDirichletCalibrator(reg_lambda=self.reg_lambda, reg_mu=self.reg_mu)
        self._model.fit(probs_val, y_val)
        return self
    def calibrate(self, probs):
        return np.clip(self._model.predict_proba(probs), EPS, 1.0 - EPS)

print("✓ CatBoost + Dirichlet redefined")

## 4. New model: MC-Dropout MLP (T=30 stochastic forward passes)

Subclasses SingleMLP. Architecture is identical (`256 → 128`, dropout=0.1).
The only change is at inference time: BatchNorm uses running stats (eval mode), but
dropout layers are kept active and we average over T=30 forward passes. This is
Gal & Ghahramani's MC-Dropout as Bayesian approximation.

In [ ]:
import torch.nn as nn

class MCDropoutMLP(SingleMLP):
    """SingleMLP with MC-Dropout at inference: T forward passes with dropout active."""

    def __init__(self, T=30, **kwargs):
        super().__init__(**kwargs)
        self.T = T

    def predict_proba(self, X):
        assert self._model is not None
        # Set everything to eval (so BatchNorm uses running stats)
        self._model.eval()
        # ...then flip dropout layers back to train mode (so they stay active)
        for module in self._model.modules():
            if isinstance(module, nn.Dropout):
                module.train()

        with torch.no_grad():
            Xt = torch.FloatTensor(X).to(self.device)
            probs_samples = []
            for _ in range(self.T):
                logits = self._model(Xt)
                probs_samples.append(torch.softmax(logits, dim=-1).cpu().numpy())

        return np.mean(np.stack(probs_samples, axis=0), axis=0)

# Smoke test
from sklearn.datasets import make_classification
Xtr, ytr = make_classification(n_samples=200, n_features=10, n_classes=3, n_informative=5, random_state=0)
m = MCDropoutMLP(T=30, input_dim=10, n_classes=3, epochs=20, batch_size=32, device=get_device())
m.fit(Xtr, ytr, Xtr[:50], ytr[:50], seed=0)
p1 = m.predict_proba(Xtr[:5])
p2 = m.predict_proba(Xtr[:5])
# With dropout active, two forward passes should differ slightly
print(f"✓ MC-Dropout smoke test. Shape: {p1.shape}")
print(f"  Predictions differ across calls (dropout active): {not np.allclose(p1, p2, atol=1e-6)}")

## 5. New calibrator: member-level temperature scaling

For ensembles. Fits a separate temperature T_m on each member's predictions over Xval_cal, then averages the per-member-scaled softmax outputs at test time. This is the non-approximate version of TS for ensembles (vs. the pseudo-logit approximation used in your paper, which TS-scales the post-averaged probabilities).

Implemented as a function rather than a `BaseCalibrator` subclass because it requires per-member predictions of shape `(M, n, K)`, not the standard `(n, K)`.

In [ ]:
import torch.optim as optim

def fit_temperature_scalar(probs, y, lr=0.01, max_iter=200):
    """Fit a single temperature T > 0 on (clipped log-probs, labels) via L-BFGS."""
    log_probs = np.log(np.clip(probs, EPS, 1.0)).astype(np.float32)
    logits_t  = torch.from_numpy(log_probs)
    labels_t  = torch.from_numpy(y.astype(np.int64))

    log_T = nn.Parameter(torch.zeros(1))
    optimizer = optim.LBFGS([log_T], lr=lr, max_iter=max_iter)
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        scaled = logits_t / log_T.exp()
        loss = criterion(scaled, labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_T.exp().item())


def apply_member_level_ts(probs_val_cal_members, probs_test_members, y_val_cal):
    """Fit T_m per member on Xval_cal probs, apply each T_m to its test probs,
    average across members. Returns (calibrated_test_probs, list_of_T_m)."""
    M = probs_val_cal_members.shape[0]
    scaled_members = []
    Ts = []

    for m in range(M):
        T_m = fit_temperature_scalar(probs_val_cal_members[m], y_val_cal)
        Ts.append(T_m)

        # Apply T_m: softmax(log_probs / T_m)
        log_probs = np.log(np.clip(probs_test_members[m], EPS, 1.0))
        scaled = log_probs / T_m
        shifted = scaled - scaled.max(axis=1, keepdims=True)
        exp_s = np.exp(shifted)
        scaled_members.append(exp_s / exp_s.sum(axis=1, keepdims=True))

    avg = np.mean(np.stack(scaled_members, axis=0), axis=0)
    return np.clip(avg, EPS, 1.0), Ts

print("✓ member-level TS function defined")

## 6. Refactored runner with member-probability caching for ensembles

Two prob-cache formats now coexist in `WORK_DIR/probs/`:
- `<task>__<model>__seed<s>.npz` — averaged probs only (Part 1 format)
- `<task>__<model>__seed<s>__members.npz` — averaged + member probs (Part 2 format for ensembles)

When we need member probs for an ensemble, we look for the `__members` file. If absent (e.g. for the M=5 ensemble whose Part 1 cache lacks member probs), we train fresh and save the `__members` file. 

In [ ]:
import time

ENSEMBLE_SIZES = {"deep_ensemble": 5, "deep_ensemble_m3": 3, "deep_ensemble_m10": 10}

def build_model(model_name, n_classes, n_features, seed, device):
    if model_name == "lgbm":        return LightGBMModel(n_classes=n_classes), {}
    if model_name == "xgboost":     return XGBoostModel(n_classes=n_classes), {}
    if model_name == "catboost":    return CatBoostModel(n_classes=n_classes), {}
    if model_name == "single_mlp":
        return SingleMLP(input_dim=n_features, n_classes=n_classes,
                         hidden_dims=(256, 128), lr=1e-3, epochs=200,
                         batch_size=256, patience=20, device=device), {"seed": seed}
    if model_name == "mc_dropout":
        return MCDropoutMLP(T=30, input_dim=n_features, n_classes=n_classes,
                            hidden_dims=(256, 128), lr=1e-3, epochs=200,
                            batch_size=256, patience=20, device=device), {"seed": seed}
    if model_name in ENSEMBLE_SIZES:
        return DeepEnsemble(n_members=ENSEMBLE_SIZES[model_name],
                            input_dim=n_features, n_classes=n_classes,
                            hidden_dims=(256, 128), lr=1e-3, epochs=200,
                            batch_size=256, patience=20, device=device), {"base_seed": seed}
    raise ValueError(f"Unknown model: {model_name}")


def build_calibrator_extended(name):
    if name == "dirichlet":     return DirichletODIRCalibrator()
    return build_calibrator(name)


def probs_path(task_id, model_name, seed):
    return PROBS_DIR / f"{task_id}__{model_name}__seed{seed}.npz"

def members_path(task_id, model_name, seed):
    return PROBS_DIR / f"{task_id}__{model_name}__seed{seed}__members.npz"


def get_predictions(split, model_name, seed, device, need_members=False):
    """Train-or-load predictions. If need_members=True (member-level TS), ensures
    per-member probs are available (re-trains if cached file lacks them)."""
    task_id = split["task_id"]
    is_ensemble = model_name in ENSEMBLE_SIZES

    # Try member-prob cache first if we need it (or if it exists for ensembles)
    if is_ensemble:
        mpath = members_path(task_id, model_name, seed)
        if mpath.exists():
            data = np.load(mpath)
            return {
                "probs_val_cal":         data["probs_val_cal"],
                "probs_test":            data["probs_test"],
                "probs_val_cal_members": data["probs_val_cal_members"],
                "probs_test_members":    data["probs_test_members"],
                "y_val_cal":             data["y_val_cal"],
                "y_test":                data["y_test"],
                "train_time_s":          float(data["train_time_s"]),
            }
        # else fall through to either standard cache (if member probs not needed) or fresh training

    standard_path = probs_path(task_id, model_name, seed)
    if standard_path.exists() and not (is_ensemble and need_members):
        data = np.load(standard_path)
        return {
            "probs_val_cal": data["probs_val_cal"],
            "probs_test":    data["probs_test"],
            "y_val_cal":     data["y_val_cal"],
            "y_test":        data["y_test"],
            "train_time_s":  float(data["train_time_s"]),
        }

    # Train fresh
    model, fit_kwargs = build_model(model_name, split["n_classes"], split["n_features"], seed, device)
    t0 = time.time()
    model.fit(split["X_train"], split["y_train"], split["X_val_es"], split["y_val_es"], **fit_kwargs)
    train_time = time.time() - t0

    probs_val_cal = model.predict_proba(split["X_val_cal"])
    probs_test    = model.predict_proba(split["X_test"])

    out = {
        "probs_val_cal": probs_val_cal,
        "probs_test":    probs_test,
        "y_val_cal":     split["y_val_cal"],
        "y_test":        split["y_test"],
        "train_time_s":  train_time,
    }

    if is_ensemble:
        probs_vc_m = model.predict_proba_members(split["X_val_cal"])
        probs_te_m = model.predict_proba_members(split["X_test"])
        out["probs_val_cal_members"] = probs_vc_m
        out["probs_test_members"]    = probs_te_m
        np.savez_compressed(
            members_path(task_id, model_name, seed),
            probs_val_cal=probs_val_cal, probs_test=probs_test,
            probs_val_cal_members=probs_vc_m, probs_test_members=probs_te_m,
            y_val_cal=split["y_val_cal"], y_test=split["y_test"],
            train_time_s=np.array(train_time),
        )
    else:
        np.savez_compressed(
            standard_path,
            probs_val_cal=probs_val_cal, probs_test=probs_test,
            y_val_cal=split["y_val_cal"], y_test=split["y_test"],
            train_time_s=np.array(train_time),
        )

    return out


def evaluate_cell(split, model_name, calibrator_name, seed, preds):
    """Apply calibrator (regular or member-level) to cached preds, compute metrics."""
    y_test = preds["y_test"]
    y_vc   = preds["y_val_cal"]
    probs_test    = preds["probs_test"]
    probs_val_cal = preds["probs_val_cal"]

    if calibrator_name == "temp_member":
        # Requires member probs
        assert "probs_val_cal_members" in preds, f"temp_member needs member probs, missing for {model_name}"
        probs_cal, _ = apply_member_level_ts(
            preds["probs_val_cal_members"], preds["probs_test_members"], y_vc
        )
        cal_label = "Member-level Temperature Scaling"
    else:
        calibrator = build_calibrator_extended(calibrator_name)
        calibrator.fit(probs_val_cal, y_vc)
        probs_cal = calibrator.calibrate(probs_test)
        cal_label = (CALIBRATOR_LABELS.get(calibrator_name)
                     if calibrator_name in CALIBRATOR_LABELS
                     else "Dirichlet ODIR")

    metrics     = evaluate_all(probs_cal,  y_test)
    raw_metrics = evaluate_all(probs_test, y_test)

    return {
        "task_id":          split["task_id"],
        "dataset_name":     split["name"],
        "n_samples":        split["n_samples"],
        "n_train":          split["n_train"],
        "n_val_es":         split["n_val_es"],
        "n_val_cal":        split["n_val_cal"],
        "n_test":           split["n_test"],
        "n_features":       split["n_features"],
        "n_classes":        split["n_classes"],
        "seed":             seed,
        "model":            model_name,
        "calibrator":       calibrator_name,
        "calibrator_label": cal_label,
        **{f"cal_{k}": v for k, v in metrics.items()},
        **{f"raw_{k}": v for k, v in raw_metrics.items()},
        "train_time_s":     round(preds["train_time_s"], 2),
    }

print("✓ Refactored runner with member-prob support defined")

## 7. Configuration 

In [ ]:
TASK_IDS = manifest["task_id"].tolist()
SEEDS    = [0, 1, 2, 3, 4]
assert len(TASK_IDS) == 36

# All 5 regular calibrators for the 3 new models
REGULAR_CALIBRATORS = ["none", "temp", "logistic", "isotonic", "dirichlet"]

PART2_CELLS = []
# M=3 ensemble: all 5 regular calibrators + member-level TS
for c in REGULAR_CALIBRATORS + ["temp_member"]:
    PART2_CELLS.append(("deep_ensemble_m3", c))
# M=10 ensemble: same
for c in REGULAR_CALIBRATORS + ["temp_member"]:
    PART2_CELLS.append(("deep_ensemble_m10", c))
# M=5 ensemble: only member-level TS (regular 5 already in Part 1)
PART2_CELLS.append(("deep_ensemble", "temp_member"))
# MC-Dropout: all 5 regular calibrators
for c in REGULAR_CALIBRATORS:
    PART2_CELLS.append(("mc_dropout", c))

device = get_device()
print(f"Device: {device}")
print(f"New (model, calibrator) cells per (dataset, seed): {len(PART2_CELLS)}")
total_target = len(PART2_CELLS) * len(SEEDS) * len(TASK_IDS)
print(f"Target new rows: {total_target}")

# Build existing_keys from Part 1 results
existing_keys = set()
for _, row in existing.iterrows():
    existing_keys.add((int(row["task_id"]), row["model"], row["calibrator"], int(row["seed"])))
print(f"Existing rows (Part 1): {len(existing_keys)}")

# Which cells actually need computing (excluding any already in CSV)
cells_to_compute = []
for task_id in TASK_IDS:
    for seed in SEEDS:
        for model_name, cal_name in PART2_CELLS:
            key = (int(task_id), model_name, cal_name, int(seed))
            if key not in existing_keys:
                cells_to_compute.append(key)
print(f"Cells to actually compute: {len(cells_to_compute)}")

## 8. Run benchmark

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

partial_ckpt = WORK_DIR / "part2_partial.csv"
if partial_ckpt.exists():
    prev = pd.read_csv(partial_ckpt)
    new_results = prev.to_dict("records")
    done_keys = {(int(r["task_id"]), r["model"], r["calibrator"], int(r["seed"]))
                 for r in new_results}
    print(f"Resumed: {len(new_results)} rows already done in this run.")
else:
    new_results = []
    done_keys = set()

# Build map: (task, model, seed) → list of calibrators to compute for that cell
from collections import defaultdict
work_plan = defaultdict(list)
for (task_id, model_name, cal_name, seed) in cells_to_compute:
    key = (task_id, model_name, cal_name, seed)
    if key in done_keys:
        continue
    work_plan[(task_id, seed, model_name)].append(cal_name)

print(f"Unique (task, seed, model) to process: {len(work_plan)}")
fail_log = []
processed = len(done_keys)
target = len(cells_to_compute)
start = time.time()

# Iterate by dataset for clean checkpointing
for task_id in TASK_IDS:
    data = None
    for seed in SEEDS:
        for model_name in ["deep_ensemble_m3", "deep_ensemble_m10", "deep_ensemble", "mc_dropout"]:
            cals = work_plan.get((task_id, seed, model_name), [])
            if not cals:
                continue

            if data is None:
                data = load_task(task_id)
                if data is None:
                    print(f"  ⚠ task {task_id} load failed")
                    fail_log.append({"task_id": task_id, "reason": "load_task None"})
                    break
                print(f"\n── Task {task_id}: {data['name']} (n={data['n_samples']}) ──")
            split = split_dataset(data, seed=seed)

            need_members = "temp_member" in cals
            try:
                preds = get_predictions(split, model_name, seed, device, need_members=need_members)
            except Exception as exc:
                print(f"    ✗ {model_name} seed={seed} TRAIN FAILED: {exc}")
                fail_log.append({"task_id": task_id, "model": model_name, "seed": seed,
                                 "reason": f"train: {exc}"})
                continue

            for cal_name in cals:
                try:
                    row = evaluate_cell(split, model_name, cal_name, seed, preds)
                    new_results.append(row)
                    done_keys.add((task_id, model_name, cal_name, seed))
                    processed += 1
                except Exception as exc:
                    print(f"    ✗ {model_name}/{cal_name} seed={seed} CAL FAILED: {exc}")
                    fail_log.append({"task_id": task_id, "model": model_name,
                                     "seed": seed, "calibrator": cal_name,
                                     "reason": f"calibrate: {exc}"})

    if new_results:
        pd.DataFrame(new_results).to_csv(partial_ckpt, index=False)
    elapsed = time.time() - start
    pct = 100 * processed / max(target, 1)
    eta = elapsed / max(processed, 1) * (target - processed) if processed > 0 else 0
    print(f"  ✓ task done. {processed}/{target} ({pct:.1f}%) | elapsed {elapsed/60:.1f}min | ETA {eta/60:.1f}min")

total_min = (time.time() - start) / 60
print(f"\n══════════════════════════════════════")
print(f"DONE. New rows: {len(new_results)} (target {target})")
print(f"Failures: {len(fail_log)}")
print(f"Total wall-clock: {total_min:.1f} min")
if fail_log:
    pd.DataFrame(fail_log).to_csv(WORK_DIR / "part2_failures.csv", index=False)

## 9. Merge 

In [ ]:
new_df = pd.DataFrame(new_results)
print(f"New rows: {len(new_df)}")
print(f"Existing rows: {len(existing)}")

merged = pd.concat([existing, new_df], ignore_index=True, sort=False)
merged = merged.drop_duplicates(subset=["task_id","model","calibrator","seed"], keep="last")
print(f"\nMerged rows: {len(merged)} (target: {len(existing)+len(new_df)})")

# Backup Part 1 version
from datetime import datetime
backup = WORK_DIR / f"results_raw_after_part1_{datetime.now():%Y%m%d_%H%M%S}.csv"
existing.to_csv(backup, index=False)
print(f"✓ Part 1 backup: {backup.name}")

merged.to_csv(WORK_DIR / "results_raw.csv", index=False)
print(f"✓ Final merged: results_raw.csv ({len(merged)} rows)")

# Clean up
if partial_ckpt.exists():
    partial_ckpt.unlink()

# Sanity matrix
print("\nRow count by (model, calibrator):")
print(merged.groupby(["model","calibrator"]).size().unstack(fill_value=0).to_string())

In [ ]:
# Quick headline preview — does ensemble size matter?
two_stage = (merged
    .groupby(["task_id","model","calibrator"])
    [["cal_nll","cal_ece_mean","cal_brier_score"]]
    .median()
    .reset_index())

medians = (two_stage.groupby(["model","calibrator"])
    [["cal_nll","cal_ece_mean","cal_brier_score"]]
    .median().round(4))

print("=== Per-(model, calibrator) median across 36 datasets ===")
ens_models = ["deep_ensemble_m3", "deep_ensemble", "deep_ensemble_m10"]
print("\n── Ensemble size sweep under matched calibration ──")
for cal in ["none", "temp", "isotonic", "dirichlet", "temp_member"]:
    if not all((m, cal) in medians.index for m in ens_models):
        continue
    nlls  = [medians.loc[(m, cal), "cal_nll"]  for m in ens_models]
    eces  = [medians.loc[(m, cal), "cal_ece_mean"] for m in ens_models]
    print(f"  {cal:15s}  NLL: M3={nlls[0]:.4f}  M5={nlls[1]:.4f}  M10={nlls[2]:.4f}  "
          f"|  ECE: M3={eces[0]:.4f}  M5={eces[1]:.4f}  M10={eces[2]:.4f}")

print("\n── Member-level vs pseudo-logit TS for each ensemble ──")
for m in ens_models:
    if (m, "temp") in medians.index and (m, "temp_member") in medians.index:
        for met in ["cal_nll", "cal_ece_mean"]:
            t = medians.loc[(m, "temp"), met]
            tm = medians.loc[(m, "temp_member"), met]
            print(f"  {m:18s} {met:14s} pseudo={t:.4f}  member={tm:.4f}  diff={tm-t:+.4f}")

print("\n── MC-Dropout vs SingleMLP ──")
for cal in ["none", "temp", "dirichlet"]:
    if ("mc_dropout", cal) in medians.index and ("single_mlp", cal) in medians.index:
        for met in ["cal_nll", "cal_ece_mean"]:
            mlp = medians.loc[("single_mlp", cal), met]
            mcd = medians.loc[("mc_dropout", cal), met]
            print(f"  {cal:12s} {met:14s} SingleMLP={mlp:.4f}  MC-Dropout={mcd:.4f}  diff={mcd-mlp:+.4f}")